In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-05 18:40:31.089182: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-05 18:40:31.937960: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "lazy",
      "params": {
        "num_of_workers": 3,
        "ips": ['172.190.116.144', '172.190.116.144', '172.190.116.144'],
        "ports": [50151, 50152, 50153]
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 1,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-05 18:40:33,161 [ERROR] [Rain] Error in the config: Error in partitions: argument of type 'int' is not iterable
2023-07-05 18:40:33,162 [DEBUG] [Rain] Rain is initialized
2023-07-05 18:40:33,163 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 18:40:33,164 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-05 18:40:33,166 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 18:40:33,166 [DEBUG] [LazyProvisioner] Provisioner is initialized
2023-07-05 18:40:33,167 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 18:40:33,168 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 18:40:33,169 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-05 18:40:33,180 [INFO] [Provisioner] provisioner is serving
2023-07-05 18:40:33,181 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 18:40:33,183 [INFO] [Coordinator] coordinator is serving
2023-07-05 18:40:33,183 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 18:40:33,187 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-05 18:40:33,188 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 18:40:33,190 [DEBUG] [LazyProvisioner] Creating 1 workers
2023-07-05 18:40:33,191 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 18:40:33,193 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 18:40:33,194 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 18:40:33,196 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-05 18:40:33

Error in receiving the gradients from the workers: [Errno 2] No such file or directory: '../../..//RainData/divider/2_2_trained.pkl'
Error in receiving the gradients from the workers: [Errno 2] No such file or directory: '../../..//RainData/divider/3_3_trained.pkl'


2023-07-05 18:42:29,468 [DEBUG] [DividerAmbassador] Error sending the data to the worker: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "recvmsg:Connection reset by peer"
	debug_error_string = "UNKNOWN:Error received from peer  {created_time:"2023-07-05T18:42:29.467333696+03:00", grpc_status:14, grpc_message:"recvmsg:Connection reset by peer"}"
>
2023-07-05 18:42:29,472 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/1.pkl to worker1
2023-07-05 18:42:29,610 [ERROR] [DividerAmbassador] Error sending the model to the worker: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv4:172.190.116.144:50151: Failed to connect to remote host: Connection refused"
	debug_error_string = "UNKNOWN:failed to connect to all addresses; last error: UNKNOWN: ipv4:172.190.116.144:50151: Failed to connect to remote host: Connection refused {created_

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.

2023-07-05 18:43:09,740 [DEBUG] [Coordinator] coordinator is sending workers info to divider
2023-07-05 18:43:45,294 [DEBUG] [Coordinator] coordinator is sending workers info to divider
2023-07-05 18:45:47,314 [DEBUG] [Coordinator] coordinator is sending workers info to divider
2023-07-05 18:48:43,287 [DEBUG] [Coordinator] coordinator is sending workers info to divider


In [ ]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 18:05:36,302 [INFO] [Provisioner] provisioner is serving
2023-07-05 18:05:36,303 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 18:05:36,304 [INFO] [Coordinator] coordinator is serving
2023-07-05 18:05:36,305 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 18:05:36,307 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-05 18:05:36,308 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 18:05:36,309 [DEBUG] [LocalProvisioner] Creating 1 workers
2023-07-05 18:05:36,310 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 18:05:36,311 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 18:05:36,311 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 18:05:36,313 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1'], ports: [50151], statuses: [1], IDs : [1]
202

469/469 [==============================] - 3s 4ms/step - loss: 0.1936 - accuracy: 0.9427


2023-07-05 18:05:53,229 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 18:05:53,230 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider


2023-07-05 18:05:53,317 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-05 18:05:53,338 [DEBUG] [DeepLearning] Iteration 1/1 complete.
DEBUG:DeepLearning:Iteration 1/1 complete.
2023-07-05 18:05:53,341 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-05 18:05:53,342 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving
2023-07-05 18:05:53,343 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-05 18:05:53,343 [INFO] [Worker_50151] Worker stopped serving on port: 50151
INFO:Worker_50151:Worker stopped serving on port: 50151
2023-07-05 18:05:53,345 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-05 18:05:53,345 [INFO] [Worker_50151] Worker stopped serving on port: 50151
INFO:Wor

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.1033 - accuracy: 0.9700

Test accuracy: 97.0%
